# Xarray-Spatial Hydrology: Fill, flow direction, accumulation, watersheds, and stream networks

Every raindrop that lands on a landscape follows a path downhill. Given a digital elevation model, the hydrology tools in xarray-spatial reconstruct those paths computationally: filling depressions, routing flow, accumulating drainage, and extracting the stream network that results. These are foundational operations for flood modeling, erosion studies, and water resource planning.

### What you'll build

1. Detect sinks in a raw DEM
2. Fill depressions so water can drain to the edges
3. Compute D8 flow direction and compare it briefly with MFD
4. Accumulate upstream drainage area
5. Delineate drainage basins automatically
6. Extract and classify the stream network (Strahler and Shreve ordering)
7. Segment streams into individually labeled links
8. Snap pour points onto channels and delineate targeted watersheds
9. Trace downstream flow paths from arbitrary start cells

![Hydrology watershed preview](images/hydrology_watershed_preview.png)

[Sink Detection](#Sink-Detection) · [Depression Filling](#Depression-Filling) · [Flow Direction](#Flow-Direction) · [Multiple Flow Direction](#Multiple-Flow-Direction) · [Flow Accumulation](#Flow-Accumulation) · [Drainage Basins](#Drainage-Basins) · [Stream Order](#Stream-Order) · [Stream Link](#Stream-Link) · [Snap Pour Point](#Snap-Pour-Point) · [Watershed Delineation](#Watershed-Delineation) · [Flow Path Tracing](#Flow-Path-Tracing)

Standard imports plus log-scale normalization for flow accumulation maps.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, LogNorm
from matplotlib.patches import Patch

import xrspatial

## Elevation data

We load a 600x600 window from the [Copernicus 30m DEM](https://registry.opendata.aws/copernicus-dem/) covering the southern Washington Cascades. If the remote file is unavailable, we fall back to xarray-spatial's built-in terrain generator. The same DEM is reused in every section below.

In [2]:
try:
    import rasterio
    from rasterio.windows import Window

    url = (
        "https://copernicus-dem-30m.s3.amazonaws.com/"
        "Copernicus_DSM_COG_10_N46_00_W123_00_DEM/"
        "Copernicus_DSM_COG_10_N46_00_W123_00_DEM.tif"
    )

    with rasterio.open(url) as src:
        # Read a 600x600 window from the southern Cascades
        window = Window(col_off=2400, row_off=2400, width=600, height=600)
        data = src.read(1, window=window).astype(np.float64)
        nodata = src.nodata

    if nodata is not None:
        data[data == nodata] = np.nan

    H, W = data.shape
    dem = xr.DataArray(data, dims=['y', 'x'], name='elevation',
                       attrs={'res': (1, 1)})
    dem['y'] = np.linspace(H - 1, 0, H)
    dem['x'] = np.linspace(0, W - 1, W)
    print(f"Loaded Copernicus 30m DEM: {dem.shape}, "
          f"elevation range {np.nanmin(dem.values):.0f} to {np.nanmax(dem.values):.0f} m")

except Exception as e:
    print(f"Remote DEM unavailable ({e}), generating synthetic terrain")
    H, W = 600, 600
    dem = xr.DataArray(np.zeros((H, W)), dims=['y', 'x'])
    dem = dem.xrs.generate_terrain(seed=10)
    dem.name = 'elevation'
    print(f"Generated terrain: {dem.shape}")

Loaded Copernicus 30m DEM: (600, 600), elevation range 564 to 2532 m


Hillshade plus draped elevation colors. This base map reappears throughout the notebook as context for each hydrology output.

In [ ]:
hillshade = dem.xrs.hillshade()

def plot_basemap(ax):
    """Plot hillshade + elevation basemap."""
    hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
    dem.plot.imshow(ax=ax, cmap='terrain', alpha=0.5, add_colorbar=False)

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
ax.set_axis_off()

## Sink Detection

Real DEMs contain small depressions (sinks) where water pools instead of flowing to the grid boundary. The `sink` function finds cells whose [D8 direction code](https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-analyst/how-flow-direction-works.htm) is 0 (no downhill neighbor) and groups adjacent ones into labeled depressions using 8-connected component labeling.

We compute a temporary flow direction grid on the raw DEM to show where sinks are. The plot highlights depression cells in orange over the basemap.

In [ ]:
# Compute flow direction on the raw DEM to find sinks
flow_dir_raw = xrspatial.flow_direction(dem)
sinks = xrspatial.sink(flow_dir_raw)

n_sink_cells = int(np.sum(~np.isnan(sinks.values)))
n_sink_groups = len(np.unique(sinks.values[~np.isnan(sinks.values)]))
print(f"{n_sink_cells} sink cells in {n_sink_groups} depressions "
      f"({100 * n_sink_cells / (H * W):.1f}% of grid)")

sink_overlay = xr.where(np.isfinite(sinks), 1.0, np.nan)

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
sink_overlay.plot.imshow(ax=ax, cmap=ListedColormap(['darkorange']),
                         alpha=200/255, add_colorbar=False)
ax.legend(handles=[Patch(facecolor='darkorange', alpha=0.78,
                         label=f'Sink cells ({n_sink_groups} depressions)')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

## Depression Filling

The `fill` function raises each depression cell to the elevation of its spill point using the [Planchon-Darboux algorithm](https://doi.org/10.1016/S0341-8162(01)00164-3), giving every cell a continuous downhill path to the grid boundary. This is a standard DEM preprocessing step for hydrology.

Filling creates flat areas where many adjacent cells share exactly the same elevation, which leaves the D8 algorithm with no unique steepest neighbor (direction code 0). To resolve flats, we add sub-millimeter random noise after filling. The plot shows which cells were raised.

In [ ]:
dem_filled = xrspatial.fill(dem)

fill_depth = dem_filled - dem
n_filled = int(np.sum(fill_depth.values > 0))
max_depth = np.nanmax(fill_depth.values)
print(f"Filled {n_filled} cells, max fill depth: {max_depth:.2f} m")

# Resolve flats: add sub-mm noise, re-fill the micro-sinks that creates,
# then add even finer noise to break any remaining ties.
rng = np.random.RandomState(42)
dem_filled.values += rng.uniform(0, 0.001, dem_filled.shape)
dem_filled = xrspatial.fill(dem_filled)
rng2 = np.random.RandomState(123)
dem_filled.values += rng2.uniform(0, 1e-6, dem_filled.shape)

fill_overlay = xr.where(fill_depth > 0, 1.0, np.nan)

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
fill_overlay.plot.imshow(ax=ax, cmap=ListedColormap(['darkorange']),
                         alpha=200/255, add_colorbar=False)
ax.legend(handles=[Patch(facecolor='darkorange', alpha=0.78,
                         label=f'Filled cells ({n_filled})')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

<div class="alert alert-block alert-warning">
<b>Filling removes real features.</b> Lakes, wetlands, and other closed basins are legitimate depressions, not DEM errors. Use the <code>z_limit</code> parameter to cap the maximum fill depth per cell so that shallow noise gets fixed while deep depressions stay intact.
</div>

## Flow Direction

The `flow_direction` function assigns each cell a [D8 direction code](https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-analyst/how-flow-direction-works.htm) indicating which of its 8 neighbors receives all downhill flow. Codes are powers of two (1=E, 2=SE, 4=S ... 128=NE), with 0 for pits and NaN for nodata.

Here we compute flow direction on the filled DEM. This is the direction grid used by every tool from here on. The color wheel shows direction codes mapped to hue.

In [ ]:
flow_dir = xrspatial.flow_direction(dem_filled)

# Verify that filling + perturbation eliminated interior sinks
sinks_after = xrspatial.sink(flow_dir)
n_sinks_after = int(np.sum(~np.isnan(sinks_after.values)))
print(f"Sinks remaining after fill + perturbation: {n_sinks_after}")

fig, ax = plt.subplots(figsize=(10, 7.5))
flow_dir.plot.imshow(ax=ax, cmap='hsv', add_colorbar=True,
                     cbar_kwargs={'label': 'D8 direction code', 'shrink': 0.7})
ax.set_axis_off()

## Multiple Flow Direction

D8 sends all flow to a single neighbor. The `flow_direction_mfd` function partitions flow to all downslope neighbors, using an adaptive exponent from [Qin et al. (2007)](https://doi.org/10.1080/13658810601168578) that concentrates flow on steep slopes and disperses it on gentle ones. The output is a 3-D DataArray `(8, H, W)` with fractional weights that sum to 1.0 at each cell.

The plot below shows the maximum fraction per cell: values near 1.0 mean flow is concentrated (D8-like), while lower values mean it spreads across multiple neighbors. See [notebook 27](27_Stream_Analysis_Dinf_MFD.ipynb) for a deeper comparison of D8, D-infinity, and MFD routing.

In [ ]:
mfd_fracs = xrspatial.flow_direction_mfd(dem_filled)
print(f"MFD output shape: {mfd_fracs.shape}")
print(f"Dims: {mfd_fracs.dims}")

max_frac = mfd_fracs.max(dim='neighbor')

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
max_frac.plot.imshow(ax=ax, cmap='YlOrRd', alpha=200/255, add_colorbar=True,
                     cbar_kwargs={'label': 'Max flow fraction (1.0 = single receiver)',
                                  'shrink': 0.7})
ax.set_axis_off()

## Flow Accumulation

For each cell, `flow_accumulation` counts how many upstream cells drain through it (including itself). The result spans several orders of magnitude: hilltops have accumulation near 1, while river channels collect thousands of upstream cells. Log-scale rendering reveals the full drainage network.

This is the workhorse of the hydrology stack. Most downstream tools (stream order, stream link, snap pour point) take the accumulation grid as input, and thresholding it is the standard way to extract a stream network.

In [ ]:
flow_accum = xrspatial.flow_accumulation(flow_dir)

print(f"Accumulation range: {np.nanmin(flow_accum.values):.0f} to "
      f"{np.nanmax(flow_accum.values):.0f}")

water_cmap = LinearSegmentedColormap.from_list('water', ['white', '#08306b'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
flow_accum.plot.imshow(ax=ax, cmap=water_cmap,
                       norm=LogNorm(vmin=1, vmax=float(np.nanmax(flow_accum.values))),
                       add_colorbar=True,
                       cbar_kwargs={'label': 'Upstream cell count (log scale)',
                                    'shrink': 0.7})
ax.set_axis_off()

In [ ]:
# Extract a stream network by thresholding accumulation.
# Lower thresholds give denser networks; higher thresholds keep only major channels.
threshold = 200
streams = xr.where(flow_accum >= threshold, flow_accum, np.nan)

n_stream_cells = int(np.sum(~np.isnan(streams.values)))
print(f"Stream cells (accum >= {threshold}): {n_stream_cells}")

stream_cmap = LinearSegmentedColormap.from_list('stream', ['lightblue', 'darkblue'])

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
streams.plot.imshow(ax=ax, cmap=stream_cmap, alpha=220/255,
                    norm=LogNorm(vmin=threshold,
                                 vmax=float(np.nanmax(streams.values))),
                    add_colorbar=False)
ax.legend(handles=[Patch(facecolor='steelblue', alpha=0.86,
                         label=f'Stream cells (accum >= {threshold})')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

## Flow Accumulation (MFD)

`flow_accumulation_mfd` routes upstream contributing area through all downslope paths at once, using the fractional weights from `flow_direction_mfd`. Where D8 accumulation produces sharp single-pixel drainage lines, MFD accumulation spreads flow and produces smoother contributing-area fields.

The side-by-side comparison shows D8 (left) with crisp channels and MFD (right) with diffuse drainage patterns.

In [ ]:
flow_accum_mfd = xrspatial.flow_accumulation_mfd(mfd_fracs)

# Log-transform for visualization (log1p handles zeros)
log_d8 = xr.DataArray(np.log1p(flow_accum.values),
                      dims=flow_accum.dims, coords=flow_accum.coords)
log_mfd = xr.DataArray(np.log1p(flow_accum_mfd.values),
                       dims=flow_accum_mfd.dims, coords=flow_accum_mfd.coords)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

log_d8.plot.imshow(ax=axes[0], cmap='Blues', add_colorbar=True,
                   cbar_kwargs={'label': 'log(1 + accum)', 'shrink': 0.7})
axes[0].set_title('D8 flow accumulation')
axes[0].set_axis_off()

log_mfd.plot.imshow(ax=axes[1], cmap='Blues', add_colorbar=True,
                    cbar_kwargs={'label': 'log(1 + accum)', 'shrink': 0.7})
axes[1].set_title('MFD flow accumulation')
axes[1].set_axis_off()

plt.tight_layout()

## Drainage Basins

The `basin` function identifies every outlet in the D8 grid (pits and cells that flow off the edge), assigns each a unique ID, and labels every cell with the outlet it drains to. No pour points needed.

This gives a quick overview of the full drainage structure before you decide where to place pour points for more targeted watershed analysis. The colored regions below show the automatic basin delineation.

In [ ]:
basins = xrspatial.basin(flow_dir)

n_basins = len(np.unique(basins.values[~np.isnan(basins.values)]))
print(f"Found {n_basins} drainage basins")

basin_cmap = LinearSegmentedColormap.from_list(
    'basin', ['#2166ac', '#66bd63', '#fee08b', '#f46d43', '#d73027', '#7b3294'])

fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
basins.plot.imshow(ax=ax, cmap=basin_cmap, alpha=150/255, add_colorbar=False)
ax.legend(handles=[Patch(facecolor='#66bd63', alpha=0.59,
                         label=f'{n_basins} drainage basins')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

## Stream Order

The `stream_order` function classifies stream cells by their position in the drainage hierarchy. Two methods are available:

- **Strahler**: a headwater is order 1. When two streams of the same order meet, the downstream segment increments by one. Different-order confluences keep the higher order.
- **Shreve**: magnitude equals the sum of all upstream headwaters. Every confluence adds its tributaries' magnitudes.

The first plot shows Strahler ordering (thin blue = order 1 headwaters, deep blue = highest order). The second shows Shreve magnitude on a continuous scale.

In [ ]:
strahler = xrspatial.stream_order(
    flow_dir, flow_accum, threshold=200, method='strahler'
)
max_order = int(np.nanmax(strahler.values))
print(f"Max Strahler order: {max_order}")

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
strahler.plot.imshow(ax=ax, cmap=stream_cmap, alpha=220/255,
                     add_colorbar=True,
                     cbar_kwargs={'label': 'Strahler order', 'shrink': 0.7})
ax.set_axis_off()

In [ ]:
shreve = xrspatial.stream_order(
    flow_dir, flow_accum, threshold=200, method='shreve'
)
print(f"Max Shreve magnitude: {int(np.nanmax(shreve.values))}")

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
shreve.plot.imshow(ax=ax, cmap=stream_cmap, alpha=220/255,
                   add_colorbar=True,
                   cbar_kwargs={'label': 'Shreve magnitude', 'shrink': 0.7})
ax.set_axis_off()

<div class="alert alert-block alert-info">
<b>Threshold sensitivity.</b> The <code>threshold</code> parameter controls how dense the extracted network is. A low threshold (50) catches small headwater channels, a high one (500+) keeps only major rivers. There is no universally correct value. Calibrate against known stream maps or field data for your study area.
</div>

## Stream Link

The `stream_link` function assigns a unique integer ID to each contiguous stream segment between junctions, headwaters, and outlets. This is useful for per-reach statistics (length, slope, contributing area) or for building a graph data structure from the stream network.

Each color in the plot below represents a distinct stream segment.

In [ ]:
links = xrspatial.stream_link(flow_dir, flow_accum, threshold=200)

link_ids = np.unique(links.values[~np.isnan(links.values)])
print(f"Found {len(link_ids)} stream link segments")

link_cmap = LinearSegmentedColormap.from_list(
    'link', ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e'])

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
links.plot.imshow(ax=ax, cmap=link_cmap, alpha=200/255, add_colorbar=False)
ax.legend(handles=[Patch(facecolor='#1b9e77', alpha=0.78,
                         label=f'{len(link_ids)} stream segments')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

## Snap Pour Point

Hand-placed pour points rarely land exactly on the drainage channel. The `snap_pour_point` function moves each pour point to the highest flow-accumulation cell within a circular search radius, so that downstream `watershed` calls delineate correctly.

Below we place three pour points slightly off the drainage network (red dots) and snap them onto the channels (blue dots).

In [ ]:
H, W = dem.shape

pour_points = xr.DataArray(
    np.full((H, W), np.nan, dtype=np.float64),
    dims=dem.dims, coords=dem.coords,
)

# Find the peak-accumulation cell in three quadrants, then offset by
# a few pixels to simulate hand-placed pour points that miss the channel.
accum_vals = flow_accum.values.copy()
accum_vals[np.isnan(accum_vals)] = 0

quadrants = [
    (slice(20, H // 2 - 20), slice(20, W // 2 - 20)),
    (slice(20, H // 2 - 20), slice(W // 2 + 20, W - 20)),
    (slice(H // 2 + 20, H - 20), slice(20, W // 2 - 20)),
]

for label, (ys, xs) in enumerate(quadrants, start=1):
    sub = accum_vals[ys, xs]
    lr, lc = np.unravel_index(sub.argmax(), sub.shape)
    r = min(ys.start + lr + 5, H - 1)
    c = max(xs.start + lc - 3, 0)
    pour_points.values[r, c] = float(label)

snapped = xrspatial.snap_pour_point(
    flow_accum, pour_points, search_radius=10
)

# Get row, col positions for scatter plotting
pp_rows, pp_cols = np.where(~np.isnan(pour_points.values))
sn_rows, sn_cols = np.where(~np.isnan(snapped.values))

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
streams.plot.imshow(ax=ax, cmap='Blues', alpha=100/255,
                    norm=LogNorm(vmin=threshold,
                                 vmax=float(np.nanmax(streams.values))),
                    add_colorbar=False)
ax.scatter(pp_cols, pp_rows, c='red', s=60, zorder=5, edgecolors='white',
           linewidths=0.8, label='Original')
ax.scatter(sn_cols, sn_rows, c='#2166ac', s=60, zorder=5, edgecolors='white',
           linewidths=0.8, label='Snapped')
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

<div class="alert alert-block alert-warning">
<b>Search radius trade-off.</b> A large <code>search_radius</code> is more forgiving of placement error but can jump the pour point to the wrong channel entirely. If two streams run close together, keep the radius small enough that snapping stays on the intended branch.
</div>

## Watershed Delineation

The `watershed` function traces every cell downstream through the D8 direction grid until it reaches a pour point (or exits the grid). Each cell gets labeled with the pour point it drains to.

This is the targeted version of `basin`: instead of finding all outlets automatically, you supply specific pour points and get back the contributing area for each one. The plot shows the three delineated watersheds with stream channels and pour points overlaid.

In [ ]:
ws = xrspatial.watershed(flow_dir, snapped)

ws_ids = np.unique(ws.values[~np.isnan(ws.values)])
print(f"Delineated {len(ws_ids)} watersheds")

ws_cmap = LinearSegmentedColormap.from_list('ws', ['#2166ac', '#66bd63', '#fee08b'])

fig, ax = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax, cmap='gray', add_colorbar=False)
ws.plot.imshow(ax=ax, cmap=ws_cmap, alpha=150/255, add_colorbar=False)
streams.plot.imshow(ax=ax, cmap='Blues', alpha=120/255,
                    norm=LogNorm(vmin=threshold,
                                 vmax=float(np.nanmax(streams.values))),
                    add_colorbar=False)
ax.scatter(sn_cols, sn_rows, c='red', s=60, zorder=5, edgecolors='white',
           linewidths=0.8, label='Pour points')
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

# Save preview image for the "What you'll build" cell
matplotlib.use('Agg')
fig_preview, ax_preview = plt.subplots(figsize=(10, 7.5))
hillshade.plot.imshow(ax=ax_preview, cmap='gray', add_colorbar=False)
ws.plot.imshow(ax=ax_preview, cmap=ws_cmap, alpha=150/255, add_colorbar=False)
streams.plot.imshow(ax=ax_preview, cmap='Blues', alpha=120/255,
                    norm=LogNorm(vmin=threshold,
                                 vmax=float(np.nanmax(streams.values))),
                    add_colorbar=False)
ax_preview.scatter(sn_cols, sn_rows, c='red', s=60, zorder=5,
                   edgecolors='white', linewidths=0.8)
ax_preview.set_axis_off()
fig_preview.savefig('examples/user_guide/images/hydrology_watershed_preview.png',
                    bbox_inches='tight', dpi=120)
plt.close(fig_preview)
matplotlib.use('module://matplotlib_inline.backend_inline')

## Flow Path Tracing

The `flow_path` function does the opposite of `watershed`: given a set of start points, it follows the D8 direction grid downstream from each one, marking every cell along the way with that start point's label. Paths terminate at pits, NaN cells, or the grid edge.

Below we pick five tributary cells (moderate accumulation) spread across the grid and trace their downstream paths. Each color represents a different start point's flow path to the main channels and grid boundary.

In [ ]:
start_points = xr.DataArray(
    np.full((H, W), np.nan, dtype=np.float64),
    dims=dem.dims, coords=dem.coords,
)

# Pick tributary cells (moderate accumulation) spread across the grid.
accum_vals = flow_accum.values.copy()
accum_vals[np.isnan(accum_vals)] = 0
trib_mask = (accum_vals >= 50) & (accum_vals <= 200)
trib_rows, trib_cols = np.where(trib_mask)

regions = [
    (slice(20, H // 3), slice(20, W // 3)),
    (slice(20, H // 3), slice(2 * W // 3, W - 20)),
    (slice(H // 3, 2 * H // 3), slice(W // 3, 2 * W // 3)),
    (slice(2 * H // 3, H - 20), slice(20, W // 3)),
    (slice(2 * H // 3, H - 20), slice(2 * W // 3, W - 20)),
]
label = 1
for ys, xs in regions:
    in_region = (
        (trib_rows >= ys.start) & (trib_rows < ys.stop) &
        (trib_cols >= xs.start) & (trib_cols < xs.stop)
    )
    if np.any(in_region):
        idx = np.where(in_region)[0][len(np.where(in_region)[0]) // 2]
        start_points.values[trib_rows[idx], trib_cols[idx]] = float(label)
        label += 1

paths = xrspatial.flow_path(flow_dir, start_points)

n_path_cells = int(np.sum(~np.isnan(paths.values)))
n_starts = int(np.sum(~np.isnan(start_points.values)))
print(f"Traced {n_path_cells} path cells from {n_starts} start points")

path_cmap = LinearSegmentedColormap.from_list(
    'path', ['#d95f02', '#e6ab02', '#1b9e77', '#7570b3', '#e7298a'])

fig, ax = plt.subplots(figsize=(10, 7.5))
plot_basemap(ax)
paths.plot.imshow(ax=ax, cmap=path_cmap, alpha=220/255, add_colorbar=False)
ax.legend(handles=[Patch(facecolor='#d95f02', alpha=0.86,
                         label=f'{n_starts} flow paths traced')],
          loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

<div class="alert alert-block alert-danger">
<b>Coordinate order matters.</b> All hydrology tools expect <code>dims=['y', 'x']</code> with y as rows. If your DataArray has transposed dimensions, the results will be silently wrong. Check <code>your_data.dims</code> before running any flow routing.
</div>

### References

- [D8 flow direction encoding](https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-analyst/how-flow-direction-works.htm), Esri ArcGIS Pro documentation
- [Planchon-Darboux depression filling](https://doi.org/10.1016/S0341-8162(01)00164-3), Planchon & Darboux 2001
- [Adaptive MFD exponent](https://doi.org/10.1080/13658810601168578), Qin et al. 2007
- [Strahler stream ordering](https://doi.org/10.1029/TR038i006p00913), Strahler 1957
- [D-infinity flow direction](https://doi.org/10.1029/96WR03137), Tarboton 1997
- [Copernicus 30m DEM on AWS](https://registry.opendata.aws/copernicus-dem/)
- [xrspatial API docs](https://makepath.github.io/xarray-spatial/reference/)